# Setup

# Network Build — Synapse Edge Lists

**Purpose:** Builds synapse edge lists from eCREST reconstruction files. For each reconstructed
cell, reads post-synaptic and pre-synaptic annotations from `.json` files and outputs a flat
synapse dataframe.

**Inputs:**
- `EM_data_published/reconstructions_published/` — eCREST `.json` reconstruction files
- `EM_data_published/data_processed_published/df_type_auto_typed.csv` — cell type assignments

**Outputs:**
- `EM_data_published/data_processed_published/df_postsyn.csv`
- `EM_data_published/data_processed_published/df_presyn.csv`

**Used by:** `Analyses_published.ipynb` (Sensory Input, Granule Cell Axons, MG Connectivity sections)

**Launch requirement:** Run `jupyter lab` from `efish_em/Notebooks_Jupyter/`

In [ ]:
############################################################################################################################ 
# Get the latest CREST files for each ID within the target folder (dirname)

from pathlib import Path
import json
from sqlite3 import connect as sqlite3_connect
from sqlite3 import DatabaseError
from igraph import Graph as ig_Graph
from igraph import plot as ig_plot
from scipy.spatial.distance import cdist
from random import choice as random_choice
from itertools import combinations
from numpy import array, unravel_index, argmin, mean
import random
import numpy as np
from copy import deepcopy
import itertools
from datetime import datetime
from time import time
import neuroglancer
from webbrowser import open as wb_open
from webbrowser import open_new as wb_open_new
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from tqdm import tqdm


import sys
from pathlib import Path
DIR_ROOT = Path.cwd().parent / 'efish_em'
if str(DIR_ROOT) not in sys.path:
    sys.path.insert(0, str(DIR_ROOT))

# from eCREST_cli_beta import ecrest, import_settings
from eCREST_cli import ecrest
import AnalysisCode as efish 


The 'ecrest' class has been imported from eCREST_cli.py

An instance of this object will be able to:
- open an neuroglancer viewer for proofrieading (see "Proofread using CREST")
    - add-remove segments (using graph feature for efficiency)
    - format itself and save itself as a CREST-style .json
- convert from neuroglancer json (see "Convert From Neuroglancer to eCREST")
    - format itself and save itself as a CREST-style .json
    


### Import settings

If you save a copy of settings_dict.json (found in the "under construction" directory of eCREST repo) locally somewhere outside the repo (like in your save_dir), then you can use the following code cell to import. This avoids needing to re-type the save_dir and db_path each time you "git pull" updates from the repo to this notebook.

In [ ]:
# Settings are loaded via dirpath below — no settings_dict.json needed for published pipeline

In [ ]:
DATA_ROOT = Path.cwd().parent.parent / 'EM_data_published'
dirpath = DATA_ROOT / 'reconstructions_published'


# Get all cells info

In [72]:
nodefiles = efish.get_cell_filepaths(dirpath)

## Cell Types

## from file

In [73]:
df_type = pd.read_csv(dirpath.parent / 'data_processed_published/df_type_auto_typed.csv')

## Base Segments

In [74]:
# Create a base_segments dictionary of all reconstructed cells 

base_segments = {}
for x,f in nodefiles.items():
    # if cell_type[x] in network_types: # if do this, you can't check if the post-syn segments exist as a reconstruction
    cell_data = efish.load_ecrest_celldata(f) #ecrest(settings_dict,filepath = f)#,launch_viewer=False)
    base_segments[cell_data['metadata']['main_seg']['base']] = cell_data['base_segments']
    
    try:
        assert cell_data['metadata']['main_seg']['base'] == x
    except:
        print(x,cell_data['metadata']['main_seg']['base'])

## Cell structure labeling checks

In [117]:
print('need labeling for:')

for x,segs in base_segments.items():
    if (len(segs['unknown']) == len([s for k,v in segs.items() for s in v])) & (cell_type[x] in ['lf','lg']):
        print(f'{x} {cell_type[x]}')

need labeling for:


# Build Graph

In [65]:
network_types = set(['smpl'])#set([v for v in df_type['cell_type'].unique()])#['pe'] #set(['sg1','sg2','smpl','grc','mg1','mg2','lg','lf']) #set([v for v in df_type['cell_type'].unique()])#['smpl']#['tsd']# 

In [82]:
synanno_type = 'post-synaptic'
vx_sizes = [16, 16, 30]

## find edges and set the cell-structure attribute of the edge based on which part of the cell the edge goes to
edge_list = []

with tqdm(total=len(nodefiles.keys())) as pbar:
    for x_pre,f_pre in nodefiles.items():
        pbar.update(1)
        pre = efish.load_ecrest_celldata(f_pre)
        
        # if pre['metadata']['cell-type']['manual'] in network_types: 
        this_type_annotations=[]
        if synanno_type in pre['end_points'].keys():
            for pos, point in enumerate(pre['end_points'][synanno_type]):
                if point[-1] not in ['annotatePoint','annotateBoundingBox','annotateSphere']:
                    point = (point, 'annotatePoint') 
                this_type_annotations.append(point)
            # rewrite end_points for point_type with new formatting
            pre['end_points'][synanno_type] =  this_type_annotations  
        
        # if the node has post-synaptic annotations (the current cell is assumed pre-synaptic)
        # pre = ecrest(settings_dict,filepath = nodefiles[x_pre])#,launch_viewer=False)
        if pre['end_points'][synanno_type] != []:
            # for each synapse
            for syn_ in pre['end_points'][synanno_type]:
                '''assumes that the annotation is a point annotation stored in the list as ([x,y,z,segment_id],'annotatePoint')
                previous ot Jan 25 2024, it was just [x,y,z,segment_id]'''
                syn_ = syn_[0]
                try:
                    post_seg = syn_[3]
                    syn_ = array([int(syn_[i]) for i in range(3)]) # synapses annotations exported as nanometers, so do not need to convert

                    # go through each other nodes
                    for x_post in nodefiles.keys():
                        # if cell_type[x_post] in network_types:
                        post = base_segments[x_post] 
                        for k,v in post.items():
                            for v_ in list(v): #find keys (can be multiple on the same cell) for matching segment ids
                                if post_seg == v_: 
                                    # add edge to the graph between current node and matching node
                                    edge_list.append([x_pre,x_post,k,syn_[0],syn_[1],syn_[2]])
                                        

                except IndexError as msg:
                    cellid = x_pre
                    print(msg, f'for cell {cellid} synapse at {array([int(syn_[i]/vx_sizes[i]) for i in range(3)])} voxels has no segment id')

            else:
                continue


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5025/5025 [46:09<00:00,  1.81it/s]


## Specific cell(S)

In [ ]:
edge_list = []
cell_list = ['297083216']
synanno_type = 'post-synaptic'

# NOTE: This cell demonstrates single-cell edge building for debugging/testing.
# It uses the ecrest class directly and requires settings_dict to be defined.
# In the published pipeline, use the full loop above (see "Build Graph" section).
for x_pre in cell_list:
    pre = ecrest(settings_dict,filepath = nodefiles[x_pre])
    if pre.cell_data['end_points'][synanno_type] != []:
        # for each synapse
        for syn_ in pre.cell_data['end_points'][synanno_type]:
            '''assumes that the annotation is a point annotation stored in the list as ([x,y,z,segment_id],'annotatePoint')
            previous ot Jan 25 2024, it was just [x,y,z,segment_id]'''
            syn_ = syn_[0]
            try:
                post_seg = syn_[3]
                syn_ = array([int(syn_[i]) for i in range(3)]) # synapses annotations exported as nanometers, so do not need to convert

                # go through each other nodes
                for x_post in nodefiles.keys():
                    # if cell_type[x_post] in network_types:
                    post = base_segments[x_post] 
                    for k,v in post.items():
                        for v_ in list(v): #find keys (can be multiple on the same cell) for matching segment ids
                            if post_seg == v_: 
                                # add edge to the graph between current node and matching node
                                
                                edge_list.append([x_pre,x_post,k,syn_[0],syn_[1],syn_[2]])
                                    

            except IndexError as msg:
                cellid = x_pre
                print(msg, f'for cell {cellid} synapse at {array([int(syn_[i]/vx_sizes[i]) for i in range(3)])} voxels has no segment id')

    else:
        continue


# Synapses dataframe

In [83]:
df_syn = pd.DataFrame(edge_list,columns = ['pre','post','structure','x','y','z'])

# for i,r in df_syn.iterrows():
#     df_syn.loc[i,'pre_type']=cell_type[df_syn.loc[i,'pre']]
#     df_syn.loc[i,'post_type']=cell_type[df_syn.loc[i,'post']]

In [37]:
for i,r in df_syn.iterrows():
    try:
        df_syn.loc[i,'pre_type'] =df_type[df_type['id'].isin([int(r['pre'])])].cell_type.values[0]
        df_syn.loc[i,'post_type']=df_type[df_type['id'].isin([int(r['post'])])].cell_type.values[0]
    except:
        # print(r['pre'],r['post'])
        continue

In [38]:
df_edges = df_syn[['pre','post','pre_type','post_type']].value_counts().reset_index(name='weight') #'structure',

# save df_syn

In [84]:
savepath = dirpath.parent / 'data_processed_published'

df_syn.to_csv(savepath / 'df_postsyn.csv')

In [78]:
df_type = pd.read_csv(dirpath.parent / 'data_processed_published/df_type_auto_typed.csv')

no_pre = []
no_post = []
for i,r in df_syn.iterrows():
    # print(r['pre'])
    # df_syn.loc[i,'pre_type'] =df_type[df_type['id'].isin([int(r['pre'])])].cell_type.values[0]
    try:
        df_syn.loc[i,'pre_type'] =df_type[df_type['id'].isin([int(r['pre'])])].cell_type.values[0]
    except:
        no_pre.append(r['pre'])
        continue
    try:
        df_syn.loc[i,'post_type']=df_type[df_type['id'].isin([int(r['post'])])].cell_type.values[0]
    except:
        no_post.append(r['post'])
        continue